# ICT — Substrat argumentation : maintenance de la verite, du JTMS a l'ATMS (strate 6)

**Navigation** : [Index](README.md) | [<< BeliefTrajectories](ICT-Argumentation-BeliefTrajectories-Python.ipynb) | [>> QBFAcceptance](ICT-Argumentation-QBFAcceptance-Python.ipynb) | [jumeau Python](ICT-Argumentation-TruthMaintenance-Python.ipynb)

Carnet de **distillation** : le materiau source est la distillation-outil EPITA 2025 Argumentation, projet `1.4.1-JTMS` (issues EPITA #1961 phase 4, pipeline arrete le 15/09). Il complete l'instrument [`ICT-Argumentation-BeliefTrajectories-Python`](ICT-Argumentation-BeliefTrajectories-Python.ipynb) (Phase B, #7742) : la ou la Phase B suivait la **trajectoire** de croyances d'un debat sous semantique grounded, ce carnet distille les **mecanismes de revision** eux-memes — comment un systeme de croyances maintient sa coherence quand une premisse tombe. Trois objets : le JTMS (un seul etat courant, revise par evenement), l'ATMS (tous les environnements d'hypotheses d'un coup), et le pont vers l'AF de Dung deja porte par l'organe `ict.argumentation` (Phase A, #7336).

> **Statut epistemique** — **Sans verdict a ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut epistemique sera porte par la matrice le cas echeant.

> **Declaration organ-first** (regle `organ-first-implementation`) : ce carnet est une **copie pedagogique declaree** du projet EPITA `1.4.1-JTMS` (fichiers `jtms.py` et `atms.py`, mesures firsthand a la tete `487210fb` du clone local le 2026-09-22). Le port Python vit dans l'organe [`ict/truth_maintenance.py`](ict/truth_maintenance.py) ; sa **fidelite est prouvee par execution differentielle** contre le code source original : 5 corpus/cas (corpus `simple_beliefs` avec et sans premisse, cas cycle, ATMS avec purge retroactive, corpus `no_justification`) produisent des tables **byte-identiques** — les valeurs de reference de ce carnet sont citees depuis la source. Le port C# des organes vit dans ce carnet. Divergences declarees : pas de `networkx` (SCC par fermeture d'accessibilite), pas de `pyvis` (tables au lieu de HTML), pas d'emojis, nom de methode orthographie. Les **quirks mesures** de la source sont conserves et enseignes, pas corriges (sections 3 et 4).

## §1 — Setup

Port des organes de la serie dans ce carnet C# (convention de la serie, cf. Phase A/B). L'organe `ict.truth_maintenance` est le port EPITA declare ci-dessus ; `ict.argumentation` (Phase A, PR #7336 mergeee) fournit l'AF de Dung et la semantique grounded. Aucune dependance hors BCL : CPU pur, aucun aleatoire (determinisme complet par construction — il n'y a rien a seeder).

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Globalization;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// organe natif Phase A (Dung)
public class DungAF
{
    public List<int> Arguments { get; }
    public List<(int From, int To)> Attacks { get; }
    public DungAF(IEnumerable<int> arguments, IEnumerable<(int From, int To)> attacks)
    {
        Arguments = arguments.Distinct().OrderBy(x => x).ToList();
        Attacks = attacks.Where(e => Arguments.Contains(e.From) && Arguments.Contains(e.To)).ToList();
    }
    public HashSet<int> Attackers(int argument) => Attacks.Where(e => e.To == argument).Select(e => e.From).ToHashSet();
}

public static class Argumentation
{
    public static DungAF TweetyBird() => new DungAF(new[] { 0, 1, 2 }, new[] { (0, 1), (1, 0), (2, 1) });
    public static DungAF NixonDiamond() => new DungAF(new[] { 0, 1 }, new[] { (0, 1), (1, 0) });
    public static HashSet<int> GroundedExtension(DungAF af)
    {
        var inside = new HashSet<int>();
        while (true)
        {
            var outside = af.Arguments.Where(b => af.Attackers(b).Overlaps(inside)).ToHashSet();
            var next = af.Arguments.Where(c => af.Attackers(c).IsSubsetOf(outside)).ToHashSet();
            if (next.SetEquals(inside)) return next;
            inside = next;
        }
    }
    public static Dictionary<int, string> GroundedLabeling(DungAF af)
    {
        var inside = GroundedExtension(af);
        var outside = af.Arguments.Where(b => af.Attackers(b).Overlaps(inside)).ToHashSet();
        return af.Arguments.ToDictionary(a => a, a => inside.Contains(a) ? "in" : outside.Contains(a) ? "out" : "undec");
    }
}

// port EPITA 1.4.1-JTMS (this grain)
public class TMSBelief
{
    public string Name { get; }
    public bool? Valid { get; set; }
    public bool NonMonotonic { get; set; }
    public List<TMSJustification> Justifications { get; } = new List<TMSJustification>();
    public List<TMSJustification> Implications { get; } = new List<TMSJustification>();
    public TMSBelief(string name) { Name = name; }
    public override string ToString() => Name + " -> " + (Valid == null ? "UNKNOWN" : Valid == true ? "VALID" : "INVALID") + (NonMonotonic ? " (non-monotonic)" : "");
    public void ComputeTruthStatement()
    {
        if (NonMonotonic) { Valid = null; return; }
        Valid = Justifications.Any(j => j.In.All(b => b.Valid == true) && !j.Out.Any(b => b.Valid == true)) ? true : (bool?)null;
        Propagate();
    }
    public void Propagate()
    {
        foreach (var j in Implications) j.Conclusion.ComputeTruthStatement();
    }
    public void SetTruthValue(bool? value) { Valid = value; Propagate(); }
}

public class TMSJustification
{
    public List<TMSBelief> In { get; }
    public List<TMSBelief> Out { get; }
    public TMSBelief Conclusion { get; }
    public TMSJustification(List<TMSBelief> input, List<TMSBelief> output, TMSBelief conclusion)
    { In = input; Out = output; Conclusion = conclusion; }
}

public class JTMS
{
    public Dictionary<string, TMSBelief> Beliefs { get; } = new Dictionary<string, TMSBelief>();
    public bool Strict { get; }
    public JTMS(bool strict = false) { Strict = strict; }
    public void AddBelief(string name) { if (!Beliefs.ContainsKey(name)) Beliefs.Add(name, new TMSBelief(name)); }
    public void SetBeliefValidity(string name, bool? validity) { Beliefs[name].SetTruthValue(validity); }
    public void AddJustification(IEnumerable<string> input, IEnumerable<string> output, string conclusion)
    {
        var ins = input.ToList();
        var outs = output.ToList();
        foreach (var name in ins.Concat(outs).Concat(new[] { conclusion }))
        {
            if (!Beliefs.ContainsKey(name))
            {
                if (Strict) throw new KeyNotFoundException("Unknown belief: " + name);
                AddBelief(name);
            }
        }
        var j = new TMSJustification(ins.Select(n => Beliefs[n]).ToList(), outs.Select(n => Beliefs[n]).ToList(), Beliefs[conclusion]);
        j.Conclusion.Justifications.Add(j);
        j.Conclusion.ComputeTruthStatement();
        foreach (var b in j.In.Concat(j.Out)) b.Implications.Add(j);
        UpdateNonMonotonicBeliefs();
    }
    public void UpdateNonMonotonicBeliefs()
    {
        var succ = Beliefs.Keys.ToDictionary(n => n, n => new HashSet<string>());
        foreach (var belief in Beliefs.Values)
            foreach (var j in belief.Justifications)
                foreach (var premise in j.In.Concat(j.Out)) succ[premise.Name].Add(belief.Name);
        var reach = new Dictionary<string, HashSet<string>>();
        foreach (var start in succ.Keys)
        {
            var seen = new HashSet<string>();
            var front = new Stack<string>();
            front.Push(start);
            while (front.Count > 0)
                foreach (var next in succ[front.Pop()])
                    if (seen.Add(next)) front.Push(next);
            reach[start] = seen;
        }
        foreach (var name in succ.Keys)
            if (reach[name].Any(t => t != name && reach[t].Contains(name))) Beliefs[name].NonMonotonic = true;
    }
    public Dictionary<string, bool?> StatusTable() => Beliefs.ToDictionary(p => p.Key, p => p.Value.Valid);
    public string ExplainBelief(string name)
    {
        var belief = Beliefs[name];
        if (belief.Justifications.Count == 0) return "No justification";
        return string.Join("\n", belief.Justifications.Select(j =>
        {
            string Describe(TMSBelief b) => b.ToString() + " (" + (b.Valid == true ? "VALID" : "not-VALID") + ")";
            var ins = j.In.Count == 0 ? "-" : string.Join(", ", j.In.Select(Describe));
            var outs = j.Out.Count == 0 ? "-" : string.Join(", ", j.Out.Select(Describe));
            var valid = j.In.All(b => b.Valid == true) && !j.Out.Any(b => b.Valid == true);
            return "Justification :\nIN : " + ins + "\nOUT : " + outs + "\n-> Results: " + (valid ? "VALID" : "not-VALID");
        }));
    }
}

public class ATMSNode
{
    public string Name { get; }
    public bool IsAssumption { get; }
    public List<HashSet<string>> Label { get; set; } = new List<HashSet<string>>();
    public List<ATMSJustification> Justifications { get; } = new List<ATMSJustification>();
    public ATMSNode(string name, bool assumption = false)
    {
        Name = name; IsAssumption = assumption;
        if (assumption) Label.Add(new HashSet<string> { name });
    }
    public bool AddEnv(HashSet<string> env)
    {
        if (Label.Any(e => e.SetEquals(env))) return false;
        Label.Add(new HashSet<string>(env)); return true;
    }
}

public class ATMSJustification
{
    public List<ATMSNode> In { get; }
    public List<ATMSNode> Out { get; }
    public ATMSNode Conclusion { get; }
    public ATMSJustification(List<ATMSNode> input, List<ATMSNode> output, ATMSNode conclusion)
    { In = input; Out = output; Conclusion = conclusion; }
}

public class ATMS
{
    public const string CONTRADICTION = "⊥";
    public Dictionary<string, ATMSNode> Nodes { get; } = new Dictionary<string, ATMSNode>();
    public ATMS() { AddNode(CONTRADICTION); }
    public ATMSNode AddNode(string name, bool isAssumption = false)
    {
        if (!Nodes.ContainsKey(name)) Nodes.Add(name, new ATMSNode(name, isAssumption));
        return Nodes[name];
    }
    public ATMSNode AddAssumption(string name) => AddNode(name, true);
    public void AddJustification(IEnumerable<string> input, IEnumerable<string> output, string conclusion)
    {
        var j = new ATMSJustification(input.Select(n => Nodes[n]).ToList(), output.Select(n => Nodes[n]).ToList(), Nodes[conclusion]);
        j.Conclusion.Justifications.Add(j);
        var combinations = new List<HashSet<string>> { new HashSet<string>() };
        foreach (var node in j.In)
        {
            var next = new List<HashSet<string>>();
            foreach (var partial in combinations)
                foreach (var env in node.Label)
                {
                    var merged = new HashSet<string>(partial);
                    merged.UnionWith(env);
                    next.Add(merged);
                }
            combinations = next;
        }
        foreach (var env in combinations)
        {
            if (j.Out.Any(node => node.Label.Any(block => block.IsSubsetOf(env)))) continue;
            if (!IsConsistent(env)) continue;
            j.Conclusion.AddEnv(env);
            if (j.Conclusion.Name == CONTRADICTION) InvalidateEnvironment(env);
        }
    }
    public void InvalidateEnvironment(HashSet<string> env)
    {
        foreach (var node in Nodes.Values) node.Label = node.Label.Where(e => !env.IsSubsetOf(e)).ToList();
    }
    public List<HashSet<string>> GetEnvironments(string name) => Nodes[name].Label;
    public bool IsConsistent(IEnumerable<string> env)
    {
        var candidate = new HashSet<string>(env);
        return !Nodes[CONTRADICTION].Label.Any(e => e.SetEquals(candidate));
    }
}

public static class Verification
{
    public static void Assert(bool condition, string message)
    { if (!condition) throw new InvalidOperationException(message); }
    public static string PyBool(bool? value) => value == null ? "None" : value == true ? "True" : "False";
    public static string Table(IEnumerable<KeyValuePair<string, bool?>> table) => "{" + string.Join(", ", table.Select(p => "'" + p.Key + "': " + PyBool(p.Value))) + "}";
}

// basenames seulement dans les sorties (pas de chemin machine -- regle 6)
Console.WriteLine("organe argumentation    : port C# dans ce carnet");
Console.WriteLine("organe truth_maintenance: port C# dans ce carnet");
Console.WriteLine("semantiques JTMS/ATMS pretes.");

organe argumentation    : port C# dans ce carnet


organe truth_maintenance: port C# dans ce carnet


semantiques JTMS/ATMS pretes.


## §2 — Pourquoi reviser : le grounded est statique, la croyance ne l'est pas

La semantique grounded (Phase A) repond a une question **instantanee** : etant donne un AF complet, quels arguments sont acceptables ? Mais un debat reel est **sequentiel** et **revisable** : une premisse tombe (un temoin se reprend), un fait nouveau arrive, et la question devient : *que reste-t-il soutenu, et pourquoi ?* La Phase B traquait la trajectoire de la **collective** (l'ordre d'arrivee des arguments change l'extension). Ici on descend d'un cran : la **memoire de justification** elle-meme.

Un systeme de maintenance de la verite (*truth maintenance system*, Doyle 1979 ; de Kleer 1986 pour l'ATMS) enregistre pour chaque croyance **pourquoi** elle tient : une liste de justifications de la forme `in:[...] out:[...] -> conclusion`. Une croyance est valide s'il existe une justification dont toutes les premisses `in` sont valides **et** dont aucune clause `out` n'est valide. C'est cette seconde clause qui rend le systeme **non monotone** : retirer une information peut *creer* une validite (la clause `out` ne bloque plus). Toute la difference entre JTMS et ATMS tient dans la strategie de calcul : le JTMS maintient **un seul etat courant** qu'il revise evenement par evenement ; l'ATMS calcule **toutes les explications simultanement** (sous quels ensembles d'hypotheses chaque noeud tient) et represente l'incoherence comme un *nogood*. Ce carnet mesure les deux sur les corpus de la source, puis croise avec le grounded de l'organe.

In [2]:
// Rappel statique : l'organe Phase A repond instantanement.
var af = Argumentation.TweetyBird();
var labeling = Argumentation.GroundedLabeling(af);
var extension = Argumentation.GroundedExtension(af);
Verification.Assert(af.Arguments.SequenceEqual(new[] { 0, 1, 2 }), "Arguments Tweety differents de la reference citee");
Verification.Assert(af.Attacks.SequenceEqual(new[] { (0, 1), (1, 0), (2, 1) }), "Attaques Tweety differentes de la reference citee");
Verification.Assert(extension.SetEquals(new[] { 0, 2 }), "Extension grounded differente de la reference citee");
Verification.Assert(labeling.Count == 3 && labeling[0] == "in" && labeling[1] == "out" && labeling[2] == "in", "Labeling grounded different de la reference citee");
Console.WriteLine("AF Tweety : arguments [" + string.Join(", ", af.Arguments) + "] | attaques [" + string.Join(", ", af.Attacks.Select(e => "(" + e.From + ", " + e.To + ")")) + "]");
Console.WriteLine("grounded  : {" + string.Join(", ", labeling.Select(p => p.Key + ": '" + p.Value + "'")) + "}");
// 0='Tweety vole', 1='Tweety ne vole pas', 2='c'est un oiseau'
// -> 2 in (rien ne l'attaque), 1 out (attaque par 2), 0 in (son attaquant 1 est out).

AF Tweety : arguments [0, 1, 2] | attaques [(0, 1), (1, 0), (2, 1)]


grounded  : {0: 'in', 1: 'out', 2: 'in'}


**Lecture.** Le labeling `{'0': 'in', '1': 'out', '2': 'in'}` (indices selon l'ordre d'affichage) est un **fait accompli** : l'organe ne dit rien de ce qui se passe si l'argument 2 disparait, ni sous quelles hypotheses 0 serait defendable. C'est precisement l'information que le TMS rajoute : avec des justifications, la chute d'une premisse propage, et l'ATMS sait enumerer les mondes d'hypotheses. La section 5 montrera que les deux lectures se **croisent** sans se reduire l'une a l'autre.

## §3 — JTMS : un etat courant, revisé par evenement

Le corpus `simple_beliefs.json` de la source EPITA, reproduit en litteraux (il vit aussi dans les tests de l'organe) : `A` soutient `B` et `C` ; `B` avec `C` absent soutient `D` ; `C` est aussi soutenu par l'absence de `B` ; `B` par l'absence de `A`. On pose `A` comme premisse et on observe la table des etats — chaque croyance porte un etat tri-state : `True` (valide), `False` (invalide), `None` (inconnu).

In [3]:
var SIMPLE_BELIEFS = new[]
{
    (input: new[] { "A" }, output: Array.Empty<string>(), conclusion: "B"),
    (input: new[] { "B" }, output: new[] { "C" }, conclusion: "D"),
    (input: new[] { "A" }, output: Array.Empty<string>(), conclusion: "C"),
    (input: Array.Empty<string>(), output: new[] { "B" }, conclusion: "C"),
    (input: Array.Empty<string>(), output: new[] { "A" }, conclusion: "B")
};

var t = new JTMS(strict: false);
foreach (var j in SIMPLE_BELIEFS)
    t.AddJustification(j.input, j.output, j.conclusion);
t.SetBeliefValidity("A", true);   // la premisse tombee du debat

var table_avec_A = t.StatusTable();
var explication_D = t.ExplainBelief("D");
Verification.Assert(Verification.Table(table_avec_A) == "{'A': True, 'B': True, 'C': True, 'D': None}", "Table avec A differente de la reference citee");
Verification.Assert(explication_D == "Justification :\nIN : B -> VALID (VALID)\nOUT : C -> VALID (VALID)\n-> Results: not-VALID", "Explication de D differente de la reference citee");
Console.WriteLine("table (A pose)  : " + Verification.Table(table_avec_A));
Console.WriteLine();
Console.WriteLine(explication_D);

table (A pose)  : {'A': True, 'B': True, 'C': True, 'D': None}


Justification :
IN : B -> VALID (VALID)
OUT : C -> VALID (VALID)
-> Results: not-VALID


**Lecture.** `{'A': True, 'B': True, 'C': True, 'D': None}` — D reste **inconnu** : sa seule justification exige `B` valide ET `C` absent, or `C` est soutenu par `A`. La table est celle de la **source EPITA**, mesuree par execution differentielle (gate 1 des tests) : ce n'est pas une interpretation, c'est le comportement du moteur d'origine. L'explication imprimee au-dessus montre la cause ligne a ligne : la clause `OUT : C (VALID)` bloque la justification.

Retirons maintenant la premisse : `A` redevient inconnu. Prediction naive (monotone) : tout s'effondre. **Mesure :**

In [4]:
t.SetBeliefValidity("A", null);   // la premisse tombe
var table_sans_A = t.StatusTable();
var explication_B = t.ExplainBelief("B");
Console.WriteLine("table (A retire): " + Verification.Table(table_sans_A));
Console.WriteLine();
Console.WriteLine(explication_B);
// Temoins : la propagation est bien repartie depuis l'evenement, pas recalculee a la volee.
Verification.Assert(Verification.Table(table_avec_A) == "{'A': True, 'B': True, 'C': True, 'D': None}", "Table avec A differente de la reference citee");
Verification.Assert(Verification.Table(table_sans_A) == "{'A': None, 'B': True, 'C': None, 'D': True}", "Table sans A differente de la reference citee");
Verification.Assert(explication_B == "Justification :\nIN : A -> UNKNOWN (not-VALID)\nOUT : -\n-> Results: not-VALID\nJustification :\nIN : -\nOUT : A -> UNKNOWN (not-VALID)\n-> Results: VALID", "Explication de B differente de la reference citee");

table (A retire): {'A': None, 'B': True, 'C': None, 'D': True}


Justification :
IN : A -> UNKNOWN (not-VALID)
OUT : -
-> Results: not-VALID
Justification :
IN : -
OUT : A -> UNKNOWN (not-VALID)
-> Results: VALID


**Lecture — le quirk mesure de la negation-as-failure.** Avec `A` inconnu, `B` **survit** via sa seconde justification (`out:[A]` — l'absence de `A` suffit), `C` tombe, et `D` **remonte** (`B` valide, `C` absent). C'est le comportement exact de la source (gate 2), et il est pedagogiquement precis : une clause `out` se lit *negation as failure* — un etat inconnu satisfait la clause de blocage, exactement comme un etat invalide. Un systeme monotone aurait tout effondre ; le TMS, lui, **reorganise** la validite. C'est la signature de la revision non monotone, et la raison pour laquelle on ne peut pas se contenter du labeling statique de la section 2.

**Et les boucles ?** Une croyance qui ne tient que par une boucle de justifications ne doit pas etre mesuree : le moteur marque les composantes fortement connexes non triviales (`non_monotonic`) et force leur etat a `None` — un **refus de mesurer**, pas un zero (meme discipline que la Phase B, #13837).

In [5]:
var tc = new JTMS(strict: false);
tc.AddJustification(new[] { "B" }, Array.Empty<string>(), "A");   // A tient par B...
tc.AddJustification(new[] { "A" }, Array.Empty<string>(), "B");   // ...et B par A : boucle
tc.AddBelief("C");
tc.SetBeliefValidity("C", true);
var table_cycle = tc.StatusTable();
var marqueurs_cycle = tc.Beliefs.ToDictionary(p => p.Key, p => (bool?)p.Value.NonMonotonic);
Console.WriteLine("table          : " + Verification.Table(table_cycle));
Console.WriteLine("non_monotonic  : " + Verification.Table(marqueurs_cycle));
Verification.Assert(Verification.Table(table_cycle) == "{'B': None, 'A': None, 'C': True}", "Table du cycle differente de la reference citee");
Verification.Assert(tc.Beliefs["A"].NonMonotonic && tc.Beliefs["B"].NonMonotonic, "Marqueurs du cycle absents");
Verification.Assert(Verification.Table(marqueurs_cycle) == "{'B': True, 'A': True, 'C': False}", "Marqueurs differents de la reference citee");

table          : {'B': None, 'A': None, 'C': True}


non_monotonic  : {'B': True, 'A': True, 'C': False}


**Lecture.** `A` et `B` restent `None` **et** portent le marqueur `non_monotonic` — le moteur sait dire *pourquoi* il ne mesure pas (gate 3). La detection des cycles utilise une fermeture d'accessibilite (divergence declaree n. 1 : la source appelle `networkx`, la serie est numpy-only) ; le resultat differentiel est identique.

## §4 — ATMS : toutes les hypotheses d'un coup

Le JTMS repond a *un* scenario. L'ATMS (*assumption-based* TMS) repond a *tous* : chaque noeud porte une **etiquette** — l'ensemble des environnements d'hypotheses sous lesquels il tient. Une hypothese commence avec `{elle-meme}` ; chaque justification propage le **produit cartesien** des etiquettes de ses `in` (en bloquant si une etiquette d'un `out` est incluse dans l'environnement fusionne) ; la contradiction est un noeud `perp` dont chaque environnement devient un **nogood**, purge de toutes les etiquettes. Construisons l'instance de la source : hypotheses `p`, `q` ; `p -> x` ; `q et x -> y` ; `p et q -> contradiction` ; `p sans q -> z`.

In [6]:
var a = new ATMS();
a.AddAssumption("p"); a.AddAssumption("q");
foreach (var n in new[] { "x", "y", "z" }) a.AddNode(n);
a.AddJustification(new[] { "p" }, Array.Empty<string>(), "x");
a.AddJustification(new[] { "q", "x" }, Array.Empty<string>(), "y");
a.AddJustification(new[] { "p", "q" }, Array.Empty<string>(), ATMS.CONTRADICTION);
a.AddJustification(new[] { "p" }, new[] { "q" }, "z");

string AfficherEnvironnements(IEnumerable<HashSet<string>> environnements)
{
    var tuples = environnements.Select(e => e.OrderBy(x => x, StringComparer.Ordinal).ToArray())
        .OrderBy(e => string.Join("\0", e), StringComparer.Ordinal)
        .Select(e => "(" + string.Join(", ", e.Select(x => "'" + x + "'")) + (e.Length == 1 ? "," : "") + ")");
    return "[" + string.Join(", ", tuples) + "]";
}
var labels = a.Nodes.ToDictionary(p => p.Key, p => AfficherEnvironnements(p.Value.Label));
foreach (var pair in labels)
    Console.WriteLine(("'" + pair.Key + "'").PadRight(6) + " : " + pair.Value);
Verification.Assert(labels["x"] == "[('p',)]", "Etiquette x incorrecte");            // x tient sous {p}
Verification.Assert(labels["y"] == "[]", "Etiquette y incorrecte");                  // {p,q} est nogood : y n'a plus rien
Verification.Assert(labels["z"] == "[('p',)]", "Etiquette z incorrecte");            // blocage out : {q} non inclus dans {p}
Verification.Assert(labels.Count == 6 && labels[ATMS.CONTRADICTION] == "[]" && labels["p"] == "[('p',)]" && labels["q"] == "[('q',)]", "Autres etiquettes incorrectes");
Verification.Assert(a.IsConsistent(new[] { "p", "q" }), "Le nogood devrait etre consomme par la purge");

// Purge retroactive : y obtient {p,q} AVANT que la contradiction soit declaree.
var a2 = new ATMS();
a2.AddAssumption("p"); a2.AddAssumption("q"); a2.AddNode("y");
a2.AddJustification(new[] { "q", "p" }, Array.Empty<string>(), "y");
Verification.Assert(AfficherEnvironnements(a2.GetEnvironments("y")) == "[('p', 'q')]", "Environnement initial de y incorrect");
a2.AddJustification(new[] { "p", "q" }, Array.Empty<string>(), ATMS.CONTRADICTION);
var lab2 = AfficherEnvironnements(a2.GetEnvironments("y"));
Console.WriteLine("purge retroactive, y : " + lab2);
Verification.Assert(lab2 == "[]", "Purge retroactive incorrecte");

'⊥'    : []


'p'    : [('p',)]


'q'    : [('q',)]


'x'    : [('p',)]


'y'    : []


'z'    : [('p',)]


purge retroactive, y : []


**Lecture.** Toute la table est une lecture d'**explications simultanees** : `x` tient sous `{p}` ; `y` sous aucun environnement (son seul soutien exige `{p,q}`, qui est nogood) ; `z` sous `{p}` (la clause `out: q` ne bloque pas `{p}`, et inversement `{q}` serait bloque par l'etiquette de... rien — c'est l'asymetrie mesuree du blocage `out`, gate 5). La purge est **retroactive** : meme declaree apres coup, la contradiction vide les etiquettes deja posees (gate 5b). Le JTMS aurait fallu N re-executions pour explorer ces N mondes ; l'ATMS les porte en un seul calcul — c'est le cout qu'il paie en memoire.

**Quirk mesure et conserve** (gate 5c) : le purge retire le nogood de l'etiquette du noeud contradiction **lui-meme** — apres purge, `IsConsistent({p,q})` repond `True`. Le nogood n'est pas *memoire*, il est *consomme*. C'est le comportement de la source a la tete mesuree ; on l'enseigne tel quel (honnêtete de distillation) avec son nom : **perte documentee**, pas silencieusement corrigee.

## §5 — Le pont Dung <-> TMS, avec temoin negatif

Une attaque `(a, b)` de l'AF de Dung se lit dans le vocabulaire TMS : *la validite de `a` soutient le rejet de `b`* — une justification miroir `in:[a] -> non-b`. Le croisement ne **traduit** pas une semantique en l'autre : il les fait travailler ensemble. L'assertion ci-dessous est un **temoin negatif** : elle echouerait si le port divergeait de la source (les litteraux viennent de l'execution differentielle), si l'organe Phase A changeait son labeling, ou si le pont etait cable a l'envers. Toute re-execution de ce carnet re-verifie donc la **fidelite de la distillation** — c'est le garde-fou d'honnetete du grain.

In [7]:
af = Argumentation.TweetyBird();
labeling = Argumentation.GroundedLabeling(af);

var tp = new JTMS(strict: false);
foreach (var (u, v) in af.Attacks)
    tp.AddJustification(new[] { $"arg{u}" }, Array.Empty<string>(), $"non-arg{v}");   // attaque = rejet soutenu
foreach (var pair in labeling)
    if (pair.Value == "in")
    {
        tp.AddBelief($"arg{pair.Key}");
        tp.SetBeliefValidity($"arg{pair.Key}", true);
    }

var rejets = tp.StatusTable().Where(p => p.Key.StartsWith("non-", StringComparison.Ordinal)).OrderBy(p => p.Key, StringComparer.Ordinal).ToList();
var affichage_grounded = "{" + string.Join(", ", labeling.Select(p => p.Key + ": '" + p.Value + "'")) + "}";
Verification.Assert(affichage_grounded == "{0: 'in', 1: 'out', 2: 'in'}", "Labeling du pont different de la reference citee");
Verification.Assert(Verification.Table(rejets) == "{'non-arg0': None, 'non-arg1': True}", "Rejets differents de la reference citee");
Console.WriteLine("grounded (organe Phase A) : " + affichage_grounded);
Console.WriteLine("rejets soutenus (TMS)     : " + Verification.Table(rejets));

// TEMOIN NEGATIF (differential, tete EPITA 487210fb) : chaque attaquant accepte
// au sens grounded soutient bien le rejet de sa cible ; aucun autre rejet n'est soutenu.
foreach (var (u, v) in af.Attacks)
    if (labeling[u] == "in")
        Verification.Assert(tp.StatusTable()[$"non-arg{v}"] == true, $"pont casse : {u}->{v}");
var soutenus = tp.StatusTable().Where(p => p.Value == true && p.Key.StartsWith("non-", StringComparison.Ordinal)).Select(p => p.Key).ToHashSet();
var attendus = af.Attacks.Where(e => labeling[e.From] == "in").Select(e => $"non-arg{e.To}").ToHashSet();
Verification.Assert(soutenus.SetEquals(attendus), "Ensemble des rejets soutenus different de celui des attaques acceptees");
Verification.Assert(soutenus.Count == 1, "Nombre de rejets different de la reference citee");
Console.WriteLine("temoin negatif : OK — " + soutenus.Count + " rejet(s) soutenu(s), exactement ceux du grounded.");

grounded (organe Phase A) : {0: 'in', 1: 'out', 2: 'in'}


rejets soutenus (TMS)     : {'non-arg0': None, 'non-arg1': True}


temoin negatif : OK — 1 rejet(s) soutenu(s), exactement ceux du grounded.


**Lecture.** Le TMS soutient **exactement** les rejets dont l'attaquant est `in` au sens grounded — pas un de plus (la clause `out` ne fabrique rien ici), pas un de moins (la propagation suit les premisses posees). Les deux lectures se croisent : le grounded dit *qui est acceptable*, le TMS dit *quel rejet est soutenu et pourquoi* — et le Nixon diamond de l'exercice 3 montrera la limite du pont (deux arguments mutuellement attaquants : le grounded dit `undec`, et le pont TMS correspondant n'a aucune premisse a poser sans creer une boucle).

## Vérifiez votre compréhension

Trois exercices pour s'approprier le substrat. Stubs a completer (convention C.1 : le notebook s'execute de bout en bout meme exercices non completes — `return null`, pas d'erreur volontaire). Les attendus sont mesurables : chaque stub nomme la verification qui jugera la reponse.

**Methode recommandee** (celle de la serie, cf. Phase B) : **predire par ecrit avant de mesurer**. Les tables des sections 3-5 n'ont pas ete devinees — elles ont ete predites puis confrontees au moteur, et c'est l'ecart entre les deux qui enseigne. Le Nixon diamond (exercice 1) est le cas ou la prediction naive est la plus fausse : la boucle d'attaques ne devient pas une boucle de justifications, et comprendre *pourquoi* est exactement l'objectif.

In [8]:
// Exercice 1 — Le Nixon diamond en JTMS.
// Construire un JTMS ou 'non-A' est soutenu par B et 'non-B' par A
// (miroir des attaques du Nixon diamond), poser A comme premisse,
// puis PREDIRE par ecrit la table avant de la mesurer.
//
// Etape 1 : ajouter les deux justifications miroir.
// Etape 2 : poser la premisse A.
// Etape 3 : comparer prediction et t.StatusTable().
// Verification attendue : les croyances miroir ne forment PAS une
// boucle de justifications (non-A et non-B n'ont pas de justifications
// qui se soutiennent mutuellement) -- dites pourquoi en commentaire.
object exercice_1_nixon_jtms()
{
    // TODO etudiant
    return null;  // retourner la table mesuree
}
Console.WriteLine("Exercice 1 a completer — prediction puis mesure.");
var resultat_1 = exercice_1_nixon_jtms();

Exercice 1 a completer — prediction puis mesure.


In [9]:
// Exercice 2 — Etendre l'ATMS d'une hypothese r.
// Reprendre l'instance de la section 4 (p, q, x, y, z, contradiction
// sur {p,q}) et ajouter : hypothese 'r', justification (in: [q, r],
// out: [], conclusion: 'w').  PREDIRE l'etiquette de w avant de mesurer.
//
// Etape 1 : construire l'instance complete (dans l'ordre de la section 4).
// Etape 2 : ajouter r et la justification de w.
// Etape 3 : lire a.GetEnvironments("w").
// Verification attendue : l'etiquette de w est {(q, r)} -- aucun nogood
// ne couvre {q, r}.  Expliquer pourquoi le nogood {p,q} n'y change rien.
object exercice_2_atms_hypothese_r()
{
    // TODO etudiant
    return null;  // retourner l'ensemble des environnements de w
}
Console.WriteLine("Exercice 2 a completer — prediction puis mesure.");
var resultat_2 = exercice_2_atms_hypothese_r();

Exercice 2 a completer — prediction puis mesure.


In [10]:
// Exercice 3 — La limite du pont Dung <-> TMS.
// Charger Argumentation.NixonDiamond(), calculer son labeling grounded,
// construire le pont miroir (section 5) et tenter de poser UN argument
// comme premisse.
//
// Etape 1 : labeling de NixonDiamond (attendu : les deux 'undec').
// Etape 2 : pont miroir des attaques.
// Etape 3 : mesurer ce que le TMS soutient quand aucune premisse
// n'est posee, puis quand une boucle de SOUTIEN (pas d'attaque)
// remplace le cycle -- que marque non_monotonic ?
// Verification attendue : le grounded dit 'undec' et le TMS, sans
// premisse, ne soutient aucun rejet -- les deux lectures s'accordent
// sur le refus de trancher.
object exercice_3_pont_limite()
{
    // TODO etudiant
    return null;  // retourner (labeling, table_du_pont)
}
Console.WriteLine("Exercice 3 a completer — la limite du pont.");
var resultat_3 = exercice_3_pont_limite();

Exercice 3 a completer — la limite du pont.


## À retenir

*Ce carnet a distille la couche **revision** de l'argumentation : un moteur JTMS qui maintient un etat courant revisable (avec sa negation-as-failure mesuree et son refus de mesurer les boucles), un moteur ATMS qui porte tous les mondes d'hypotheses simultanement (avec sa purge retroactive et son nogood non persistant — quirk conserve, perte documentee), et un pont vers la semantique grounded de l'organe Phase A, verifie par temoin negatif. La **fidelite** n'est pas une impression : chaque valeur affichee est comparee aux references citees de la source EPITA (tete `487210fb`), et les gates 1-6 des tests de l'organe Python (`ict/tests/test_truth_maintenance.py`) rejouent ces litteraux a chaque CI.*

**Sources et provenance** : distillation-outil EPITA 2025 Argumentation, projet `1.4.1-JTMS` (`jtms.py`, `atms.py`, corpus `Beliefs/*.json`), issues EPITA #1961 phase 4 — port Python declare dans [`ict/truth_maintenance.py`](ict/truth_maintenance.py), port C# dans ce carnet. Organe Dung : `ict.argumentation` (Phase A, #7336), porte en C# dans ce carnet. Instrument voisin : [ICT-Argumentation-BeliefTrajectories-Python](ICT-Argumentation-BeliefTrajectories-Python.ipynb) (Phase B, #7742). Le **verdict de distillation** (fidele / perte documentee / perte par complaisance / divergence positive, par axe) est rendu sur le dashboard CoursIA, pas dans ce fichier.

## Pour aller plus loin

Doyle 1979 (*A truth maintenance system*, AI 12) pour le JTMS original ; de Kleer 1986 (*An assumption-based TMS*, AI 28) pour l'ATMS ; Dung 1995 (*On the acceptability of arguments*) pour la semantique grounded — les trois references que la source EPITA operationnalise.

- **[ICT-Argumentation-Capstone — le capstone de la strate 6](ICT-Argumentation-Capstone-Python.ipynb)** : les trois graines réunies — ce que la trajectoire de croyance dit, et ce qu'elle ne dit pas.
- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.
- **[Argumentation-06d — Complaisance auditée](../../SymbolicAI/ArgumentAnalysis/Argumentation-06d-Complaisance-Auditee-Python.ipynb)** : l'etiquetage IN/OUT de ce moteur y sert d'oracle a un audit de complaisance d'un modele servi — la reponse attendue se deduit du graphe de justifications, et l'on mesure si le modele la retrouve.

**Suivant — tête de la strate 6** : [→ ICT-Argumentation-Capstone](ICT-Argumentation-Capstone-Python.ipynb) — le capstone de la strate qui consume les trois graines (BeliefTrajectories, QBFAcceptance, TruthMaintenance) sur un corpus réel codé argument par argument.

**Jumeau Python** → [ICT-Argumentation-TruthMaintenance-Python](ICT-Argumentation-TruthMaintenance-Python.ipynb) — le carnet d'origine, en Python.